In [1]:
# Cleaning Script for Existing Kaggle Datasets

import pandas as pd
import numpy as np
import os

def clean_data(input_file, output_prefix="combined_dataframe"):
    """
    Clean dataset with 9 different options
    
    Parameters:
    -----------
    input_file : str
        Path to the input CSV file (e.g., 'NewDataSet/existingDataset/combined_dataframe_AAPL.csv')
    output_prefix : str
        Prefix for output files (e.g., 'AAPL', 'CNN_SP500')
    
    Returns:
    --------
    dict : Dictionary containing all 9 cleaned dataframes
    """
    
    # Check if file exists
    if not os.path.exists(input_file):
        print(f"❌ Error: File not found: {input_file}")
        return None
    
    # Get the directory and create output prefix
    output_dir = os.path.dirname(input_file)
    base_filename = os.path.basename(input_file).replace('.csv', '')
    
    df = pd.read_csv(input_file)
    print(f"📂 Input file: {input_file}")
    print(f"Original shape: {df.shape}\n")
    
    results = {}
    
    # ============= OPTION 1: Replace Zero Values with Small Noise =============
    print("📊 OPTION 1: Replace zeros with small random noise")
    df_noise = df.copy()
    numeric_cols = df_noise.select_dtypes(include=[np.number]).columns
    
    zero_count = 0
    for col in numeric_cols:
        zero_mask = (df_noise[col] == 0)
        if zero_mask.any():
            df_noise.loc[zero_mask, col] = np.random.normal(0, 0.0001, zero_mask.sum())
            zero_count += zero_mask.sum()
    
    print(f"  ✓ Replaced {zero_count} zeros total")
    output_file = os.path.join(output_dir, f"{base_filename}_noise.csv")
    df_noise.to_csv(output_file, index=False)
    print(f"  ✅ Saved: {output_file}\n")
    results['noise'] = df_noise

    # ============= OPTION 2: Add Small Epsilon to Constant Columns =============
    print("📊 OPTION 2: Add epsilon to constant columns")
    df_epsilon = df.copy()
    numeric_cols = df_epsilon.select_dtypes(include=[np.number]).columns
    epsilon = 1e-6
    
    constant_count = 0
    for col in numeric_cols:
        if df_epsilon[col].std() == 0:
            df_epsilon[col] = df_epsilon[col] + np.random.normal(0, epsilon, len(df_epsilon))
            constant_count += 1
    
    print(f"  ✓ Added epsilon to {constant_count} constant columns")
    output_file = os.path.join(output_dir, f"{base_filename}_epsilon.csv")
    df_epsilon.to_csv(output_file, index=False)
    print(f"  ✅ Saved: {output_file}\n")
    results['epsilon'] = df_epsilon

    # ============= OPTION 3: Forward Fill Then Backward Fill NaN =============
    print("📊 OPTION 3: Fill missing values")
    df_filled = df.copy()
    nan_count_before = df_filled.isna().sum().sum()
    df_filled = df_filled.fillna(method='ffill').fillna(method='bfill').fillna(0)
    nan_count_after = df_filled.isna().sum().sum()
    
    print(f"  ✓ Handled {nan_count_before} NaN values → {nan_count_after} remaining")
    output_file = os.path.join(output_dir, f"{base_filename}_filled.csv")
    df_filled.to_csv(output_file, index=False)
    print(f"  ✅ Saved: {output_file}\n")
    results['filled'] = df_filled

    # ============= OPTION 4: Interpolate Constant Values =============
    print("📊 OPTION 4: Interpolate constant columns")
    df_interp = df.copy()
    numeric_cols = df_interp.select_dtypes(include=[np.number]).columns
    
    interp_count = 0
    for col in numeric_cols:
        if df_interp[col].std() == 0 and df_interp[col].nunique() == 1:
            trend = np.linspace(0, 0.01, len(df_interp))
            df_interp[col] = df_interp[col] + trend
            interp_count += 1
    
    print(f"  ✓ Interpolated {interp_count} constant columns")
    output_file = os.path.join(output_dir, f"{base_filename}_interp.csv")
    df_interp.to_csv(output_file, index=False)
    print(f"  ✅ Saved: {output_file}\n")
    results['interp'] = df_interp

    # ============= OPTION 5: Clip Extreme Values =============
    print("📊 OPTION 5: Clip extreme values to 1st-99th percentile")
    df_clipped = df.copy()
    numeric_cols = df_clipped.select_dtypes(include=[np.number]).columns
    
    for col in numeric_cols:
        lower = df_clipped[col].quantile(0.01)
        upper = df_clipped[col].quantile(0.99)
        df_clipped[col] = df_clipped[col].clip(lower, upper)
    
    print(f"  ✓ Clipped all {len(numeric_cols)} numeric columns")
    output_file = os.path.join(output_dir, f"{base_filename}_clipped.csv")
    df_clipped.to_csv(output_file, index=False)
    print(f"  ✅ Saved: {output_file}\n")
    results['clipped'] = df_clipped

    # ============= OPTION 6: Replace Inf and NaN =============
    print("📊 OPTION 6: Replace Inf and NaN values")
    df_clean = df.copy()
    
    df_clean = df_clean.replace([np.inf, -np.inf], np.nan)
    numeric_cols = df_clean.select_dtypes(include=[np.number]).columns
    
    for col in numeric_cols:
        df_clean[col].fillna(df_clean[col].median(), inplace=True)
    
    print(f"  ✓ Replaced Inf/NaN values in {len(numeric_cols)} columns")
    output_file = os.path.join(output_dir, f"{base_filename}_clean.csv")
    df_clean.to_csv(output_file, index=False)
    print(f"  ✅ Saved: {output_file}\n")
    results['clean'] = df_clean

    # ============= OPTION 7: Log Transform + Add Constant =============
    print("📊 OPTION 7: Log transform to stabilize variance")
    df_log = df.copy()
    numeric_cols = df_log.select_dtypes(include=[np.number]).columns
    
    for col in numeric_cols:
        df_log[col] = np.log1p(np.abs(df_log[col])) * np.sign(df_log[col])
    
    print(f"  ✓ Applied log1p transform to {len(numeric_cols)} columns")
    output_file = os.path.join(output_dir, f"{base_filename}_log.csv")
    df_log.to_csv(output_file, index=False)
    print(f"  ✅ Saved: {output_file}\n")
    results['log'] = df_log

    # ============= OPTION 8: Standardize Each Column Independently =============
    print("📊 OPTION 8: Standardize by column (mean=0, std=1)")
    df_std = df.copy()
    numeric_cols = df_std.select_dtypes(include=[np.number]).columns
    
    for col in numeric_cols:
        mean = df_std[col].mean()
        std = df_std[col].std()
        if std > 0:
            df_std[col] = (df_std[col] - mean) / std
        else:
            df_std[col] = 0
    
    print(f"  ✓ Standardized {len(numeric_cols)} columns")
    output_file = os.path.join(output_dir, f"{base_filename}_std.csv")
    df_std.to_csv(output_file, index=False)
    print(f"  ✅ Saved: {output_file}\n")
    results['std'] = df_std

    # ============= OPTION 9: COMBINED APPROACH (Recommended) =============
    print("=" * 60)
    print("🎯 OPTION 9: COMBINED CLEANING (Recommended)")
    print("=" * 60)
    
    df_combined = df.copy()
    numeric_cols = df_combined.select_dtypes(include=[np.number]).columns
    
    # Step 1: Replace Inf values
    df_combined = df_combined.replace([np.inf, -np.inf], np.nan)
    print("  ✓ Replaced Inf values")
    
    # Step 2: Fill NaN with forward fill then backward fill
    df_combined = df_combined.fillna(method='ffill').fillna(method='bfill')
    print("  ✓ Filled NaN values")
    
    # Step 3: Add epsilon to constant columns
    for col in numeric_cols:
        if df_combined[col].std() == 0:
            df_combined[col] = df_combined[col] + np.random.normal(0, 1e-6, len(df_combined))
    
    print("  ✓ Added epsilon to constant columns")
    
    # Step 4: Clip extreme values
    for col in numeric_cols:
        lower = df_combined[col].quantile(0.01)
        upper = df_combined[col].quantile(0.99)
        df_combined[col] = df_combined[col].clip(lower, upper)
    
    print("  ✓ Clipped extreme values")
    
    # Step 5: Standardize
    for col in numeric_cols:
        mean = df_combined[col].mean()
        std = df_combined[col].std()
        if std > 0:
            df_combined[col] = (df_combined[col] - mean) / std
    
    print("  ✓ Standardized columns")
    
    output_file = os.path.join(output_dir, f"{base_filename}_combined.csv")
    df_combined.to_csv(output_file, index=False)
    print(f"\n✅ Saved: {output_file}")
    print(f"   Final shape: {df_combined.shape}\n")
    results['combined'] = df_combined
    
    return results

# ============= USAGE EXAMPLES =============
print("🚀 Starting data cleaning process...\n")
print("=" * 60)

# Example 1: Clean AAPL dataset
print("EXAMPLE 1: Cleaning AAPL Dataset")
print("=" * 60)
aapl_results = clean_data('NewDataSet/existingDataset/combined_dataframe_AAPL_raw.csv')


# Example 2: Clean CNN S&P 500 dataset (uncomment to use)
print("EXAMPLE 2: Cleaning CNN S&P 500 Dataset")
print("=" * 60)
cnn_results = clean_data('NewDataSet/existingDataset/combined_dataframe_CNN_SP500_raw.csv')


# Example 3: Clean My new dataset
print("EXAMPLE 3: Cleaning My new Dataset")
print("=" * 60)
aapl_results = clean_data('NewDataSet\mycreation\combined_dataframe_ASX200_raw.csv')



<>:237: SyntaxWarning: invalid escape sequence '\m'
<>:237: SyntaxWarning: invalid escape sequence '\m'
C:\Users\Tanvir\AppData\Local\Temp\ipykernel_14248\2689181832.py:237: SyntaxWarning: invalid escape sequence '\m'
  aapl_results = clean_data('NewDataSet\mycreation\combined_dataframe_ASX200_raw.csv')


🚀 Starting data cleaning process...

EXAMPLE 1: Cleaning AAPL Dataset
📂 Input file: NewDataSet/existingDataset/combined_dataframe_AAPL_raw.csv
Original shape: (2592, 84)

📊 OPTION 1: Replace zeros with small random noise
  ✓ Replaced 6544 zeros total
  ✅ Saved: NewDataSet/existingDataset\combined_dataframe_AAPL_raw_noise.csv

📊 OPTION 2: Add epsilon to constant columns
  ✓ Added epsilon to 1 constant columns


C:\Users\Tanvir\AppData\Local\Temp\ipykernel_14248\2689181832.py:47: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[ 1.25903644e-05  7.29289912e-05 -8.66204245e-05 -8.70434322e-05
 -2.85674440e-05 -5.09750044e-06  6.01891437e-05  1.71748278e-04
  1.94101025e-08 -1.80896637e-04  2.63250082e-05 -6.05335155e-05
 -2.54161904e-05  9.19138279e-07 -7.62215923e-05  2.47632630e-04
 -7.29586749e-05  2.07378505e-04 -4.21795911e-05 -1.11151783e-06
  3.94953109e-05  8.25158484e-05 -1.16181729e-04  1.79424449e-05
 -8.62863283e-05 -1.03146322e-04 -1.32777312e-04 -1.01652751e-04
  1.22568743e-05 -2.14223810e-04 -9.75722084e-05 -1.48015787e-04
  1.46810340e-04 -1.70993035e-04 -8.95249218e-05  1.09178997e-04
  2.72205444e-04  7.63817344e-05 -3.93914015e-06  6.18447825e-05
 -2.91299332e-05 -3.79999861e-05 -5.37881950e-05  8.80037995e-06
  1.79143737e-04 -8.44335156e-05 -4.42913245e-05  1.13807468e-04
 -8.00865831e-05  1.53

  ✅ Saved: NewDataSet/existingDataset\combined_dataframe_AAPL_raw_epsilon.csv

📊 OPTION 3: Fill missing values
  ✓ Handled 0 NaN values → 0 remaining
  ✅ Saved: NewDataSet/existingDataset\combined_dataframe_AAPL_raw_filled.csv

📊 OPTION 4: Interpolate constant columns
  ✓ Interpolated 1 constant columns


C:\Users\Tanvir\AppData\Local\Temp\ipykernel_14248\2689181832.py:78: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_filled = df_filled.fillna(method='ffill').fillna(method='bfill').fillna(0)


...

  ✅ Saved: NewDataSet\mycreation\combined_dataframe_ASX200_raw_interp.csv

📊 OPTION 5: Clip extreme values to 1st-99th percentile
  ✓ Clipped all 82 numeric columns
  ✅ Saved: NewDataSet\mycreation\combined_dataframe_ASX200_raw_clipped.csv

📊 OPTION 6: Replace Inf and NaN values
  ✓ Replaced Inf/NaN values in 82 columns


c:\Users\Tanvir\anaconda3\envs\DS\Lib\site-packages\numpy\lib\_function_base_impl.py:4671: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = subtract(b, a)
C:\Users\Tanvir\AppData\Local\Temp\ipykernel_14248\2689181832.py:129: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_clean[col].fillna(df_clean[col].median(), inplace=True)
C:\Users\Tanvir\AppData\Local\Temp\ipykernel_14248\2689181832.py:129: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an 

  ✅ Saved: NewDataSet\mycreation\combined_dataframe_ASX200_raw_clean.csv

📊 OPTION 7: Log transform to stabilize variance
  ✓ Applied log1p transform to 82 columns
  ✅ Saved: NewDataSet\mycreation\combined_dataframe_ASX200_raw_log.csv

📊 OPTION 8: Standardize by column (mean=0, std=1)
  ✓ Standardized 82 columns


c:\Users\Tanvir\anaconda3\envs\DS\Lib\site-packages\pandas\core\nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)
C:\Users\Tanvir\AppData\Local\Temp\ipykernel_14248\2689181832.py:183: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_combined = df_combined.fillna(method='ffill').fillna(method='bfill')


  ✅ Saved: NewDataSet\mycreation\combined_dataframe_ASX200_raw_std.csv

🎯 OPTION 9: COMBINED CLEANING (Recommended)
  ✓ Replaced Inf values
  ✓ Filled NaN values
  ✓ Added epsilon to constant columns
  ✓ Clipped extreme values
  ✓ Standardized columns

✅ Saved: NewDataSet\mycreation\combined_dataframe_ASX200_raw_combined.csv
   Final shape: (2064, 84)

